# Create Ludo Game Dataset

## Overview
This notebook generates a synthetic Ludo game dataset for training a machine learning model to predict winning combinations.

## What Was Done
- Simulated **1,000+ Ludo games** for 4 players: **Red, Green, Yellow, and Blue**
- Each player controls **4 tokens**, following standard Ludo rules:
  - All tokens begin in the home yard (**position 0**); a dice roll of **6** is required to move a token onto the board (**position 1**)
  - Active tokens advance by the dice value; overshooting position **56** causes a **bounce-back**
  - Landing on an opponent's active token **captures** it and sends it back to the home yard
  - Rolling a **6** or making a **capture** grants the player an extra turn
  - A player **wins** when all 4 of their tokens have reached **position 57** (finished)
- Simulation continues until at least **10,000 rows** of data are collected
- Each row records one turn for one player with full 4-token state information
- The final dataset is saved as `ludo_dataset.csv` in the `data file/Raw_Data/` directory

## Output
| Column | Description |
|---|---|
| `Game_ID` | Unique identifier for each simulated game |
| `Turn` | Global turn number within the game |
| `Player` | Player colour (Red, Green, Yellow, Blue) |
| `Dice_Roll` | Value rolled on the dice (1–6) |
| `Token_Moved` | Which token (1–4) was moved; 0 if no valid move was available |
| `Position_Before` | Token position before the move (0 = home, 1–56 = board, 57 = finished; NaN if no move) |
| `Position_After` | Token position after the move (same scale; NaN if no move) |
| `Tokens_Home` | Number of this player's tokens still in the home yard |
| `Tokens_Active` | Number of this player's tokens currently on the board |
| `Tokens_Finished` | Number of this player's tokens that have completed the circuit |
| `Captured_Opponent` | 1 if the move captured an opponent token, 0 otherwise |
| `Is_Winner` | 1 if this player won the game, 0 otherwise |


In [ ]:
import csv
import math
import random
from pathlib import Path

PLAYER_NAMES = ("Red", "Green", "Yellow", "Blue")
FINISH_POS = 57
BOARD_MAX = 56

COLUMNS = [
    "Game_ID",
    "Turn",
    "Player",
    "Dice_Roll",
    "Token_Moved",
    "Position_Before",
    "Position_After",
    "Tokens_Home",
    "Tokens_Active",
    "Tokens_Finished",
    "Captured_Opponent",
    "Is_Winner",
]


def _next_position(position: int, dice_roll: int) -> int:
    target = position + dice_roll
    if target > FINISH_POS:
        return FINISH_POS - (target - FINISH_POS)
    return target


def _valid_moves(tokens: list[int], dice_roll: int) -> list[tuple[int, int, int]]:
    moves: list[tuple[int, int, int]] = []
    for idx, pos in enumerate(tokens):
        if pos == FINISH_POS:
            continue
        if pos == 0:
            if dice_roll == 6:
                moves.append((idx, 0, 1))
            continue
        moves.append((idx, pos, _next_position(pos, dice_roll)))
    return moves


def _choose_move(rng: random.Random, moves: list[tuple[int, int, int]]) -> tuple[int, int, int]:
    if len(moves) == 1:
        return moves[0]

    best_after = max(m[2] for m in moves)
    shortlist = [m for m in moves if m[2] == best_after]
    return shortlist[rng.randrange(len(shortlist))]


def _count_token_states(tokens: list[int]) -> tuple[int, int, int]:
    home = 0
    active = 0
    finished = 0
    for pos in tokens:
        if pos == 0:
            home += 1
        elif pos == FINISH_POS:
            finished += 1
        else:
            active += 1
    return home, active, finished


def _simulate_game(game_id: int, rng: random.Random) -> list[list[object]]:
    tokens_by_player = [[0, 0, 0, 0] for _ in range(4)]
    rows: list[list[object]] = []

    turn = 1
    player_idx = 0
    winner_idx = -1

    randint = rng.randint

    while winner_idx == -1:
        dice_roll = randint(1, 6)
        current_tokens = tokens_by_player[player_idx]
        moves = _valid_moves(current_tokens, dice_roll)

        token_moved = 0
        pos_before: object = math.nan
        pos_after: object = math.nan
        captured = 0

        if moves:
            token_idx, before, after = _choose_move(rng, moves)
            token_moved = token_idx + 1
            pos_before = before
            pos_after = after
            current_tokens[token_idx] = after

            if 1 <= after <= BOARD_MAX:
                for opp_idx in range(4):
                    if opp_idx == player_idx:
                        continue
                    opp_tokens = tokens_by_player[opp_idx]
                    for j, opp_pos in enumerate(opp_tokens):
                        if opp_pos == after:
                            opp_tokens[j] = 0
                            captured = 1

            if all(pos == FINISH_POS for pos in current_tokens):
                winner_idx = player_idx

        tokens_home, tokens_active, tokens_finished = _count_token_states(current_tokens)

        rows.append(
            [
                game_id,
                turn,
                PLAYER_NAMES[player_idx],
                dice_roll,
                token_moved,
                pos_before,
                pos_after,
                tokens_home,
                tokens_active,
                tokens_finished,
                captured,
                0,
            ]
        )

        extra_turn = (dice_roll == 6) or (captured == 1)
        if not extra_turn and winner_idx == -1:
            player_idx = (player_idx + 1) % 4

        turn += 1

    winner_name = PLAYER_NAMES[winner_idx]
    for row in rows:
        row[-1] = 1 if row[2] == winner_name else 0

    return rows


def generate_dataset(
    min_rows: int = 11000,
    seed: int = 42,
    raw_output: Path = Path("../data file/Raw_Data/ludo_dataset.csv"),
    clean_output: Path = Path("../data file/Raw_Data/ludo_dataset_cleaned.csv"),
) -> int:
    min_rows = max(min_rows, 11000)
    rng = random.Random(seed)

    raw_output.parent.mkdir(parents=True, exist_ok=True)
    clean_output.parent.mkdir(parents=True, exist_ok=True)

    total_rows = 0
    game_id = 0

    with raw_output.open("w", newline="", encoding="utf-8") as raw_file:
        writer = csv.writer(raw_file)
        writer.writerow(COLUMNS)

        while total_rows < min_rows:
            game_rows = _simulate_game(game_id, rng)
            writer.writerows(game_rows)
            total_rows += len(game_rows)
            game_id += 1

    clean_output.write_bytes(raw_output.read_bytes())
    return total_rows


rows_written = generate_dataset(min_rows=11000, seed=42)
print(f"Generated {rows_written} rows")
print("Saved to ../data file/Raw_Data/ludo_dataset.csv")
print("Saved to ../data file/Raw_Data/ludo_dataset_cleaned.csv")